# Análise Exploratória dos Dados — Dados Brutos

Exploração inicial do dataset de **5.8 milhões de voos domésticos dos EUA em 2015**, proveniente do US Department of Transportation.

Objetivo: entender a estrutura dos dados, identificar padrões de valores ausentes, e explorar distribuições e relações antes da limpeza.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
sns.set_style('whitegrid')

In [ ]:
flights = pd.read_csv("../data/flights.csv", low_memory=False)
airports = pd.read_csv("../data/airports.csv")
airlines = pd.read_csv("../data/airlines.csv")

# Join airlines (by AIRLINE code)
flights = flights.merge(airlines, left_on="AIRLINE", right_on="IATA_CODE", how="left") \
                 .drop(columns="IATA_CODE") \
                 .rename(columns={"AIRLINE_y": "AIRLINE_NAME", "AIRLINE_x": "AIRLINE"})

# Join airports for ORIGIN
flights = flights.merge(airports, left_on="ORIGIN_AIRPORT", right_on="IATA_CODE", how="left") \
                 .drop(columns="IATA_CODE") \
                 .rename(columns={
                     "AIRPORT": "ORIGIN_AIRPORT_NAME",
                     "CITY": "ORIGIN_CITY",
                     "STATE": "ORIGIN_STATE",
                     "LATITUDE": "ORIGIN_LAT",
                     "LONGITUDE": "ORIGIN_LON"
                 }).drop(columns=["COUNTRY"])

# Join airports for DESTINATION
flights = flights.merge(airports, left_on="DESTINATION_AIRPORT", right_on="IATA_CODE", how="left") \
                 .drop(columns="IATA_CODE") \
                 .rename(columns={
                     "AIRPORT": "DEST_AIRPORT_NAME",
                     "CITY": "DEST_CITY",
                     "STATE": "DEST_STATE",
                     "LATITUDE": "DEST_LAT",
                     "LONGITUDE": "DEST_LON"
                 }).drop(columns=["COUNTRY"])

print(f"Shape após joins: {flights.shape}")

print(flights.head(5))

## Fontes de Dados

O dataset é composto por 3 arquivos CSV:
- **flights.csv** — 5.819.079 registros de voos com horários, atrasos e cancelamentos
- **airports.csv** — metadados dos aeroportos (nome, cidade, estado, coordenadas)
- **airlines.csv** — códigos e nomes das companhias aéreas

Os joins (left) adicionam 10 colunas de contexto geográfico. Cerca de **8,3% dos aeroportos** usam códigos FAA numéricos que não possuem correspondência na tabela IATA, gerando NaN nas colunas de join.

In [ ]:
flights.info(memory_usage='deep')

In [ ]:
missing = flights.isnull().sum()
missing_pct = (missing / len(flights) * 100).round(2)
missing_df = pd.DataFrame({'Ausentes': missing, '%': missing_pct}).query('Ausentes > 0').sort_values('%', ascending=False)
print(f"Colunas com valores ausentes: {len(missing_df)} de {len(flights.columns)}\n")
missing_df

## Estratégia de Valores Ausentes

A análise revela **4 padrões distintos** de dados faltantes:

| Padrão | Colunas | % NaN | Causa |
|--------|---------|-------|-------|
| Causas de atraso | AIR_SYSTEM_DELAY, SECURITY_DELAY, etc. | ~82% | **By design** — só preenchido para voos atrasados |
| Cancelamento | CANCELLATION_REASON | ~98,5% | **By design** — só preenchido para voos cancelados |
| Dados operacionais | DEPARTURE_DELAY, ARRIVAL_DELAY, etc. | ~1,5% | **Realmente faltante** — voos desviados ou falha de registro |
| Colunas de join | ORIGIN_AIRPORT_NAME, ORIGIN_CITY, etc. | ~8,3% | Aeroportos com código FAA numérico sem match IATA |

**Decisões para a limpeza:**
1. Causas de atraso → imputar com 0 (sem atraso = zero contribuição)
2. Voos cancelados → remover (cancelamento ≠ atraso)
3. Sem DEPARTURE_DELAY → remover (~1,5%)
4. Colunas de join → preencher com "Desconhecido"

In [ ]:
flights.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

flights['DEPARTURE_DELAY'].dropna().clip(-30, 120).hist(bins=60, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição de DEPARTURE_DELAY (clipped)')
axes[0].set_xlabel('Minutos')
axes[0].axvline(x=15, color='red', linestyle='--', alpha=0.7, label='Limiar 15 min')
axes[0].legend()

flights['ARRIVAL_DELAY'].dropna().clip(-30, 120).hist(bins=60, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Distribuição de ARRIVAL_DELAY (clipped)')
axes[1].set_xlabel('Minutos')

plt.tight_layout()
plt.show()

In [ ]:
cancel_rate = flights['CANCELLED'].mean() * 100
print(f"Taxa de cancelamento: {cancel_rate:.2f}% ({flights['CANCELLED'].sum():,} voos)\n")

cancel_map = {'A': 'Companhia Aérea', 'B': 'Clima', 'C': 'NAS (Sistema)', 'D': 'Segurança'}
cancelled_flights = flights[flights['CANCELLED'] == 1]
reason_counts = cancelled_flights['CANCELLATION_REASON'].map(cancel_map).value_counts()

reason_counts.plot(kind='bar', figsize=(8, 4), title='Razões de Cancelamento', color='salmon', edgecolor='white')
plt.ylabel('Quantidade')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(reason_counts)

In [ ]:
avg_delay_airline = flights.groupby('AIRLINE_NAME')['DEPARTURE_DELAY'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
avg_delay_airline.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Atraso Médio de Partida por Companhia Aérea')
plt.ylabel('Minutos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
month_delay = flights.groupby('MONTH')['DEPARTURE_DELAY'].mean()

plt.figure(figsize=(10, 4))
month_delay.plot(kind='bar', color='teal', edgecolor='white')
plt.title('Atraso Médio de Partida por Mês')
plt.ylabel('Minutos')
plt.xlabel('Mês')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
day_labels = {1: 'Seg', 2: 'Ter', 3: 'Qua', 4: 'Qui', 5: 'Sex', 6: 'Sáb', 7: 'Dom'}
dow_delay = flights.groupby('DAY_OF_WEEK')['DEPARTURE_DELAY'].mean()
dow_delay.index = dow_delay.index.map(day_labels)

plt.figure(figsize=(9, 4))
dow_delay.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Atraso Médio de Partida por Dia da Semana')
plt.ylabel('Minutos')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
hour_delay = flights.groupby(flights['SCHEDULED_DEPARTURE'] // 100)['DEPARTURE_DELAY'].mean()

plt.figure(figsize=(14, 4))
hour_delay.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Atraso Médio de Partida por Hora do Dia')
plt.xlabel('Hora (0–23)')
plt.ylabel('Minutos')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
airport_delay = (flights.groupby('ORIGIN_AIRPORT')['DEPARTURE_DELAY']
                 .mean()
                 .sort_values(ascending=False)
                 .head(15))

code_to_name = flights.drop_duplicates('ORIGIN_AIRPORT').set_index('ORIGIN_AIRPORT')['ORIGIN_AIRPORT_NAME']
airport_delay.index = airport_delay.index.map(lambda x: code_to_name.get(x, x))

plt.figure(figsize=(10, 6))
airport_delay.plot(kind='barh', color='coral', edgecolor='white')
plt.title('Top 15 Aeroportos por Atraso Médio de Partida')
plt.xlabel('Atraso Médio (minutos)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
state_delay = (flights[flights['ORIGIN_STATE'].notna()]
               .groupby('ORIGIN_STATE')['DEPARTURE_DELAY']
               .mean()
               .sort_values(ascending=False)
               .head(10))

plt.figure(figsize=(10, 5))
state_delay.plot(kind='barh', color='teal', edgecolor='white')
plt.title('Top 10 Estados por Atraso Médio de Partida')
plt.xlabel('Atraso Médio (minutos)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = ['DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'AIR_SYSTEM_DELAY',
                'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY',
                'WEATHER_DELAY', 'DISTANCE', 'ELAPSED_TIME']
corr = flights[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Matriz de Correlação — Variáveis de Atraso')
plt.tight_layout()
plt.show()

In [ ]:
delay_cause_cols = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
                    'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']
delayed = flights[flights['DEPARTURE_DELAY'] > 0]
cause_means = delayed[delay_cause_cols].mean().sort_values(ascending=False)

cause_means.plot(kind='bar', figsize=(10, 4),
                 title='Média de Minutos de Atraso por Causa (voos atrasados)',
                 color='orange', edgecolor='white')
plt.ylabel('Minutos Médios')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
extreme = (flights['DEPARTURE_DELAY'] > 300).sum()
max_delay = flights['DEPARTURE_DELAY'].max()
print(f"Voos com atraso de partida > 300 min: {extreme:,}  ({extreme/len(flights)*100:.2f}%)")
print(f"Atraso máximo de partida: {max_delay:.0f} min (~{max_delay/60:.1f} horas)")

## Conclusões da Exploração

### Padrões identificados
- **Sazonalidade**: Verão (Jun–Ago) e feriados de inverno (Dez–Jan) apresentam atrasos elevados; outono é o melhor período
- **Padrão diário**: Atrasos se acumulam ao longo do dia — voos matutinos são mais pontuais, voos noturnos sofrem mais
- **Dia da semana**: Quinta e sexta-feira tendem a ter os maiores atrasos; sábado é geralmente o melhor dia
- **Companhias**: Variação significativa entre carriers — regionais e low-cost tendem a atrasar mais
- **Causas dominantes**: LATE_AIRCRAFT_DELAY e AIRLINE_DELAY são os maiores contribuintes
- **Outliers**: <1% dos voos têm atrasos extremos (>300 min), sugerindo disrupções sistêmicas ocasionais
- **Imbalance**: ~82% dos voos são pontuais (atraso ≤ 15 min)

### Decisões de limpeza necessárias
→ Ver notebook **`limpeza.ipynb`** para o pipeline completo de preparação dos dados.